# Эксперимент 2: log-cosh регрессия на L1-шаре

Гладкая часть (как в прошлых лабах): средний **log-cosh** остатка `Ax - b` без L2-штрафа (`regcoef=0`),
ограничение $\|x\|_1 \le R$. Сравнение FW и Away-step FW.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents][:10]:
        if (p / "src" / "optimization.py").is_file():
            return p
    return here


ROOT = _find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.oracles import LogCoshL2Oracle
from src.optimization import away_step_frank_wolfe_method, frank_wolfe_method
from src.paths import figs_dir

FIG = figs_dir()
np.random.seed(2)

In [ ]:
m, n = 200, 50
rng = np.random.default_rng(0)
A = rng.standard_normal((m, n)) / np.sqrt(n)
x_true = np.zeros(n)
x_true[:8] = rng.standard_normal(8)
x_true = 1.4 * x_true / np.linalg.norm(x_true, 1)
b = A @ x_true + 0.15 * rng.standard_normal(m)


def matvec_Ax(x):
    return A @ np.asarray(x, dtype=float).ravel()


def matvec_ATx(u):
    return A.T @ np.asarray(u, dtype=float).ravel()


def matmat_ATsA(s):
    return A.T @ (np.asarray(s, dtype=float).reshape(-1, 1) * A)


oracle = LogCoshL2Oracle(matvec_Ax, matvec_ATx, matmat_ATsA, b, regcoef=0.0)

x0 = np.zeros(n)
R = 2.0
tol = 1e-2
max_iter = 12000

x_fw, msg_fw, h_fw = frank_wolfe_method(
    oracle, x0, R, tolerance=tol, max_iter=max_iter, step_size_strategy="armijo", trace=True
)
x_afw, msg_afw, h_afw = away_step_frank_wolfe_method(
    oracle, x0, R, tolerance=tol, max_iter=max_iter, step_size_strategy="armijo", trace=True
)

print("FW:", msg_fw, "f ≈", oracle.func(x_fw), "||x||_1", np.linalg.norm(x_fw, 1), "итераций", len(h_fw["func"]))
print("AFW:", msg_afw, "f ≈", oracle.func(x_afw), "||x||_1", np.linalg.norm(x_afw, 1), "итераций", len(h_afw["func"]))

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(h_fw["fw_gap"], label="FW gap (Frank–Wolfe)")
plt.semilogy(h_afw["fw_gap"], label="FW gap (Away-step FW)")
plt.xlabel("Итерация")
plt.ylabel("Frank–Wolfe gap")
plt.title("Log-cosh регрессия на L1-шаре: gap")
plt.legend()
plt.grid(True, which="both", ls=":", alpha=0.6)
plt.tight_layout()
out = FIG / "exp02_logcosh_fw_gap.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print("Сохранено:", out)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(h_fw["func"], label="FW")
plt.plot(h_afw["func"], label="AFW")
plt.xlabel("Итерация")
plt.ylabel("f(x_k)")
plt.title("Log-cosh: значение целевой функции")
plt.legend()
plt.grid(True, ls=":", alpha=0.6)
plt.tight_layout()
out = FIG / "exp02_logcosh_objective.png"
plt.savefig(out, dpi=160, bbox_inches="tight")
plt.show()
print("Сохранено:", out)